In [2]:
import arcpy

COLOR_TABLE = {
    1:  (255, 0,   0),
    2:  (255, 97,  27),
    3:  (248, 224, 25),
    4:  (170, 110, 40),
    5:  (240, 50,  230),
    6:  (0,   254, 36),
    7:  (24,  242, 254),
    8:  (210, 245, 60),
    9:  (94,  121, 253),
    10: (0,   0,   128),
    11: (226, 29,  253),
    12: (255, 255, 255),
}

def _to_int_value(v):
    if v is None:
        return None
    if isinstance(v, (list, tuple)) and len(v) > 0:
        v = v[0]
    try:
        return int(v)
    except Exception:
        try:
            return int(float(str(v)))
        except Exception:
            return None

def _set_item_color(item, rgb):
    r, g, b = rgb
    # 常見兩種路徑（不同 Pro 版本/不同 item 物件）
    try:
        item.color = {'RGB': [r, g, b, 100]}
        return True
    except Exception:
        pass
    try:
        item.symbol.color = {'RGB': [r, g, b, 100]}
        return True
    except Exception:
        pass
    return False

def apply_uv_colors_keep_order_labels(layer, color_table):
    """
    不改 renderer/colorizer 既有的順序與 label，只對存在的 value 套色。
    raster -> sym.colorizer
    feature -> sym.renderer
    """
    sym = layer.symbology

    # 1) 取得 raster colorizer 或 feature renderer
    if hasattr(sym, "colorizer"):         # raster layer
        core = sym.colorizer
    elif hasattr(sym, "renderer"):        # feature layer
        core = sym.renderer
    else:
        raise RuntimeError("這個 layer.symbology 既沒有 colorizer 也沒有 renderer，無法處理。")

    # 2) 走 groups/items（Unique values 常見都在這個結構）
    if not hasattr(core, "groups"):
        raise RuntimeError("此 symbology 物件找不到 groups；可能不是 Unique Values（或版本差異）。")

    changed = 0
    matched = set()

    for grp in core.groups:
        if not hasattr(grp, "items"):
            continue
        for it in grp.items:
            # 不同版本欄位名不同，依序嘗試抓 value
            raw_val = None
            for key in ("values", "value", "classValue", "gridcode", "GRIDCODE", "VALUE"):
                if hasattr(it, key):
                    raw_val = getattr(it, key)
                    break

            v_int = _to_int_value(raw_val)
            if v_int is None:
                continue

            if v_int in color_table:
                if _set_item_color(it, color_table[v_int]):
                    changed += 1
                    matched.add(v_int)

    # 3) 套回 layer
    layer.symbology = sym
    return changed, sorted(matched)

# ========= 使用：用 layer name 取得 raster layer 後套色 =========
aprx = arcpy.mp.ArcGISProject("CURRENT")
m = aprx.activeMap

target_layer_name = "OL_P1_2204_coral_label_6x6m"  # 改成你的 raster layer 名稱
lyr = next((l for l in m.listLayers() if l.name == target_layer_name), None)
if lyr is None:
    raise RuntimeError(f"找不到圖層：{target_layer_name}")

n_changed, matched_values = apply_uv_colors_keep_order_labels(lyr, COLOR_TABLE)
print(f"[OK] 已更新顏色的類別數：{n_changed}")
print(f"[OK] 成功匹配到並套色的 value：{matched_values}")


[OK] 已更新顏色的類別數：9
[OK] 成功匹配到並套色的 value：[1, 2, 3, 6, 7, 9, 10, 11, 12]
